In [1]:
!pip install sentence-transformers IsoScore

## Registery example

> In simple terms, a registry is a dictionary that maps a string to a class. Used to get the class from a string name.

In [6]:
from registry import Registry

ANIMALS = Registry("animal")

@ANIMALS.register("dog")
class Dog:
    def speak(self):
        return "woof"

print(ANIMALS.names())                  # ['dog']
print(ANIMALS.get("dog"))               # <class '__main__.Dog'>
print(Dog.name)                         # dog
print(ANIMALS.get("dog")().speak())     # woof   <- get the class, () creates an object
try:
    ANIMALS.get("cat")                   # KeyError: 'cat'
except KeyError as e:
    print(e)

['dog']
<class '__main__.Dog'>
dog
woof
"Unknown animal 'cat'. Available: ['dog']"


## Set the HF Token

In [4]:
import os
from kaggle_secrets import UserSecretsClient

# Retrieve the secret from Kaggle
user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

# Set it as an environment variable
os.environ["HF_TOKEN"] = HF_TOKEN

## Run the gemma embedding model

In [5]:
"""
IsoScore for google/embeddinggemma-300m.

Computes:
  1. Sentence-level IsoScore on the final pooled embeddings (what model.encode returns)
  2. Token-level IsoScore after every transformer layer (the setup in your Metrics section)

IsoScore uses the official implementation by Rudman et al. (2022):
  pip install IsoScore sentence-transformers
"""

import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from IsoScore import IsoScore

def isoscore(points: np.ndarray) -> float:
    """points: (num_samples, dim). Returns IsoScore in [0, 1] via the official library.
    Cast to float64 so results don't depend on the input dtype."""
    X = np.asarray(points, dtype=np.float64)
    if X.shape[0] < 2:
        raise ValueError("Need at least 2 points.")
    return float(IsoScore.IsoScore(X))


def isotropic_ceiling(n: int, d: int) -> float:
    """Approximate IsoScore a perfectly isotropic cloud would get with only n points."""
    return n / (n + d)


model = SentenceTransformer("google/embeddinggemma-300m")

sentences = [
    "That is a happy person",
    "That is a happy dog",
    "That is a very happy person",
    "Today is a sunny day",
]

# ---------------------------------------------------------------
# 1. Sentence-level IsoScore (pooled, projected, normalized output)
# ---------------------------------------------------------------
embeddings = model.encode(sentences)  # shape (4, 768)
n, d = embeddings.shape
print(f"Sentence embeddings: n={n}, d={d}")
print(f"  IsoScore            = {isoscore(embeddings):.6f}")
print(f"  Isotropic ceiling   ~ {isotropic_ceiling(n, d):.6f}  (max achievable with n={n})")

# ---------------------------------------------------------------
# 2. Token-level IsoScore after each transformer layer
#    Tokens from ALL sentences are pooled into one point cloud per layer.
# ---------------------------------------------------------------
backbone = model[0].auto_model          # underlying Gemma3 encoder
tokenizer = model.tokenizer
device = model.device

features = model.preprocess(sentences)
print(features.values())
features = {k: v.to(device) for k, v in features.items() if isinstance(v, torch.Tensor)}

with torch.no_grad():
    out = backbone(
        input_ids=features["input_ids"],
        attention_mask=features["attention_mask"],
        output_hidden_states=True,
    )

# Keep real tokens only: drop padding and special tokens (BOS/EOS carry outlier norms)
input_ids = features["input_ids"]
special_ids = torch.tensor(tokenizer.all_special_ids, device=device)
keep = features["attention_mask"].bool() & ~torch.isin(input_ids, special_ids)

print(f"\nToken-level (pooled real tokens: {int(keep.sum())})")
print(f"{'layer':>5} | {'IsoScore':>10} | {'ceiling':>8}")
# hidden_states[0] = embedding layer output, [i] = after transformer layer i
for layer_idx, hs in enumerate(out.hidden_states):
    pts = hs[keep].float().cpu().numpy()   # (num_tokens, hidden_dim)
    print(f"{layer_idx:>5} | {isoscore(pts):>10.6f} | "
          f"{isotropic_ceiling(*pts.shape):>8.4f}")

modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

2_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

Sentence embeddings: n=4, d=768
  IsoScore            = 0.001005
  Isotropic ceiling   ~ 0.005181  (max achievable with n=4)
ValuesView({'input_ids': tensor([[    2,  6372,   563,   496,  5293,  1589,     1,     0],
        [    2,  6372,   563,   496,  5293,  4799,     1,     0],
        [    2,  6372,   563,   496,  1401,  5293,  1589,     1],
        [    2, 17076,   563,   496, 17635,  1719,     1,     0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 0],
        [1, 1, 1, 1, 1, 1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 0]]), 'modality': 'text'})

Token-level (pooled real tokens: 21)
layer |   IsoScore |  ceiling
    0 |   0.007755 |   0.0266
    1 |   0.006326 |   0.0266
    2 |   0.003578 |   0.0266
    3 |   0.004444 |   0.0266
    4 |   0.003665 |   0.0266
    5 |   0.004081 |   0.0266
    6 |   0.003740 |   0.0266
    7 |   0.004567 |   0.0266
    8 |   0.004173 |   0.0266
    9 |   0.005381 |   0.0266
   10 |   0.006974 |   0.0266
   11 |  

## Dataset loading

In [ ]:
from datasets import load_dataset

ds = load_dataset("ai4bharat/IN22-Gen")

In [3]:
from dataset_loader import IN22Gen
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file
in22gen = IN22Gen()

lang_name = "hin_Deva"
in22gen.load(lang_name)

data/train-00000-of-00001.parquet:   0%|          | 0.00/4.34M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1024 [00:00<?, ? examples/s]

in22-gen/test: 1024 rows kept out of 1024 (100.0%)


['सेवा संबंधी लोगों के लिए भेष कई गुणों का संयोजन है, जैसे कि उनके जूते, कपड़े, टाई, आभूषण, केश शैली, मेक-अप, घड़ी, कॉस्मेटिक, इत्र, आदि।',
 'महाराष्ट्र के औरंगाबाद जिले में स्थित अजंता में उन्तीस चैत्य और विहार गुफाएँ हैं जो पहली शताब्दी ई.पू. से ले कर पाँचवीं शताब्दी ईस्वी तक की मूर्तियों तथा चित्रकारियों से सुसज्जित हैं।',
 'विस्तार का असर बनाते हुए, शरीर का रंग बाहरी रेखा में घुल-मिल जाता है।',
 'अशोक ने व्यापक रूप से मूर्तियों और शानदार स्मारकों को बनाने के लिए पत्थर का प्रयोग करना शुरू किया, जबकि उससे पहले पारंपरिक रूप से लकड़ी और मिट्टी का प्रयोग किया जाता है।',
 'महाराष्ट्र के इस स्वादिष्ट और प्रसिद्ध व्यंजन में आलुओं में मसाले को मिलाकर, बेसन के घोल की परत लगाकर, उसे अच्छी तरह से तल कर बनाया जाता है।',
 'चेट्टिनाड भोजन तमिल नाडु के चेट्टिनाड क्षेत्र के नाट्टूकोटाई चेट्टियार नामक समुदाय या खुद को नागारथार कहने वालों का भोजन है।',
 'दालचीनी मूल रूप से भारत की है और इस रूप में इसका प्रयोग स्वादिष्टकारक से लेकर औषधीय कारक के रूप में किया जाता है।',
 'सूखे ग्राइंडर में या फिर पत्थर

In [4]:
from dataset_loader import IN22Gen
from dotenv import load_dotenv

load_dotenv()  # Load environment variables from .env file
in22gen = IN22Gen()

lang_name = "ben_Beng"
in22gen.load(lang_name)

in22-gen/test: 1024 rows kept out of 1024 (100.0%)


['বাহ্য রূপ হল সেই পরিষেবা প্রদানকারী মানুষ সম্বন্ধীয় এক গুচ্ছ বৈশিষ্ট্য যেমন, তার জুতো, জামাকাপড়, টাই, গয়না, চুলের ধরন, রূপসজ্জা, ঘড়ি, প্রসাধনী, সুগন্ধি, ইত্যাদি।',
 'মহারাষ্ট্রের ঔরঙ্গাবাদ জেলায় অবস্থিত অজন্তায় খ্রিষ্টপূর্ব প্রথম শতাব্দী থেকে খ্রিষ্টীয় পঞ্চম শতাব্দীর ভাস্কর্য এবং চিত্রকলায় সজ্জিত ঊনত্রিশটি চৈত্য এবং বিহার গুহা রয়েছে।',
 'গায়ের রঙের সঙ্গে দেহরেখা মিশে গিয়ে ঘনত্বের আভাস তৈরি করে।',
 'যেখানে পূর্ববর্তী ঐতিহ্য ছিল কাঠ ও মাটি দিয়ে নির্মাণ করা, অশোক সেখানে ভাস্কর্য এবং মহান স্মৃতিস্তম্ভ নির্মাণের কাজে ব্যাপকভাবে পাথরের ব্যবহার শুরু করেন।',
 'আলুতে মশলা মাখিয়ে, ফেটানো বেসনে চুবিয়ে নিয়ে ডুবো তেলে ভাজা যতক্ষণ না ভালো করে ভাজা হচ্ছে, তৈরি করা হয় মহারাষ্ট্রের এই সুস্বাদু ও জনপ্রিয় খাবারের পদ।',
 'চেট্টিনাড় রন্ধনপ্রণালী হলো তামিলনাড়ু রাজ্যের চেট্টিনাড় অঞ্চলের নাট্টুকোটাই চেট্টিয়ার, অথবা তাদের নিজেদের দেওয়া নাম অনুযায়ী নগরত্তার নামক একটি সম্প্রদায়ের এক রন্ধনপ্রণালী।',
 'দারুচিনি ভারতে মূলভাবে উৎপাদিত এবং এর ব্যবহার গন্ধবিশিষ্ট দ্রব্য থেকে আরম্ভ করে ঔষধি দ্রব্য হিসেবে হয়।',
 'এই স